/Volumes/workspace/default/w2day2

create data

In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


write as delta table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/w2day2")

###read delta table

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/w2day2")
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


###insert

In [0]:
new_data = [(4, "D", 8000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/workspace/default/w2day2/")

###update

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/w2day2")

delta_table.update(
    condition = "id = 2",
    set = {"salary": "6500"}
)

DataFrame[num_affected_rows: bigint]

### delete

In [0]:
delta_table.delete("id = 1")

DataFrame[num_affected_rows: bigint]

merge into

### existing data

In [0]:
target_df = spark.read.format("delta").load("/Volumes/workspace/default/w2day2")
target_df.display()

id,name,salary
2,B,6500
3,C,7000
4,D,8000


### new data

In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,name,salary
2,B,7000
5,E,9000


### merge

### upsert

In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### new coloumn arrives

In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]

df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()

id,name,salary,country
6,F,10000,India


### enable schema evolution

In [0]:
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/w2day2/")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/workspace/default/w2day2/")
df_read.display()

id,name,salary,country
6,F,10000,India
3,C,7000,null
4,D,8000,null
2,B,7000,null
5,E,9000,null


### time travel and history viewing

In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-14T09:31:04.000Z,70834001490012,poornalakshmi135@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2757373243767417),f7a22c7e-994a-48c1-86e5-bcdea3eb8b0c,0414-085155-z65ui38a-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1199)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-14T09:14:50.000Z,70834001490012,poornalakshmi135@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2757373243767417),8454cbc7-e07f-4187-838d-7e5cbdf9aceb,0414-085155-z65ui38a-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3053, p25FileSize -> 1083, numDeletionVectorsRemoved -> 1, minFileSize -> 1083, numAddedFiles -> 1, maxFileSize -> 1083, p75FileSize -> 1083, p50FileSize -> 1083, numAddedBytes -> 1083)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-14T09:14:48.000Z,70834001490012,poornalakshmi135@gmail.com,MERGE,"Map(predicate -> [""(id#11946L = id#11949L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2757373243767417),8454cbc7-e07f-4187-838d-7e5cbdf9aceb,0414-085155-z65ui38a-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2008, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4593, materializeSourceTimeMs -> 420, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1677, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2354)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-14T09:11:55.000Z,70834001490012,poornalakshmi135@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2757373243767417),8e0f43c5-01b7-4fcb-b235-edadd6ad76ac,0414-085155-z65ui38a-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1061, p25FileSize -> 1045, numDeletionVectorsRemoved -> 1, minFileSize -> 1045, numAddedFiles -> 1, maxFileSize -> 1045, p75FileSize -> 1045, p50FileSize -> 1045, numAddedBytes -> 1045)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-14T09:11:54.000Z,70834001490012,poornalakshmi135@gmail.com,DELETE,"Map(predicate -> [""(id#11631L = 1)""])",null,List(2757373243767417),8e0f43c5-01b7-4fcb-b235-edadd6ad76ac,0414-085155-z65ui38a-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2266, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1458, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 807)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-14T09:02:49.000Z,70834001490012,poornalakshmi135@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2757373243767417),15364f8b-ba4e-4444-bf80-b97fd08a7d06,0414-085155-z65ui38a-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3051, p25FileSize -> 1061, numDeletionVectorsRemoved -> 1, minFileSize -> 1061, numAddedFiles -> 1, maxFileSize -> 1061, p75FileSize -> 1061, p50FileSize -> 1061, numAddedBytes -> 1061)",null,Databricks-Runtime/18.1.x-aar

###read old version

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/w2day2/")
df_old.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### restore table

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]